# Aula 11 - Notebook: Modelagem Topológica de Tubulações como Dígrafos Ponderados

Neste notebook implementamos a classe base `GrafoTubulacao` para representar a malha hidráulica da Estação de Reabastecimento de Hidrogênio como um Grafo Dirigido e Ponderado $G=(V, E, W)$.


In [ ]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

def formatar_matriz(matriz, rotulos_linhas, rotulos_cols):
    """Formata matriz 2D em tabela ASCII."""
    larguras = [max(len(str(r)), 6) for r in rotulos_cols]
    larg_linha = max(len(str(r)) for r in rotulos_linhas)
    header = f"{' ' * larg_linha} | " + " | ".join(f"{c:>{larguras[j]}}" for j, c in enumerate(rotulos_cols))
    divisor = f"{'-' * larg_linha}-+-" + "-+-".join("-" * larguras[j] for j in range(len(rotulos_cols)))
    linhas = [header, divisor]
    for i, r_nome in enumerate(rotulos_linhas):
        vals = []
        for j in range(len(rotulos_cols)):
            v = matriz[i][j]
            v_str = "∞" if v == float('inf') else str(v)
            vals.append(f"{v_str:>{larguras[j]}}")
        linhas.append(f"{r_nome:<{larg_linha}} | " + " | ".join(vals))
    return "\n".join(linhas)

from typing import List, Dict, Tuple, Any

class GrafoTubulacao:
    def __init__(self, vertices: List[str]):
        self.vertices = vertices
        self.v_to_idx = {v: i for i, v in enumerate(vertices)}
        self.idx_to_v = {i: v for i, v in enumerate(vertices)}
        self.n = len(vertices)
        
        self.adj_binaria = [[0] * self.n for _ in range(self.n)]
        self.adj_pesos = [[float('inf')] * self.n for _ in range(self.n)]
        for i in range(self.n):
            self.adj_pesos[i][i] = 0.0
        
        self.arestas_detalhes: List[Dict[str, Any]] = []

    def adicionar_tubulacao(self, origem: str, destino: str, comprimento_m: float, 
                           tag_valvula: str, diametro_pol: float = 1.0):
        u = self.v_to_idx[origem]
        v = self.v_to_idx[destino]
        
        self.adj_binaria[u][v] = 1
        self.adj_pesos[u][v] = comprimento_m
        
        self.arestas_detalhes.append({
            "Origem": origem,
            "Destino": destino,
            "Comprimento (m)": comprimento_m,
            "Válvula ISA": tag_valvula,
            "Diâmetro (pol)": diametro_pol
        })

    def obter_graus(self) -> List[Dict[str, Any]]:
        graus = []
        for i, v in enumerate(self.vertices):
            deg_out = sum(self.adj_binaria[i])
            deg_in = sum(self.adj_binaria[r][i] for r in range(self.n))
            graus.append({"Equipamento": v, "Grau Entrada (deg-)": deg_in, "Grau Saída (deg+)": deg_out})
        return graus

nos_processo = ["E-101", "C-101", "C-102", "TK-101", "TK-102", "CH-101", "MAN-101", "D-101"]
rede = GrafoTubulacao(nos_processo)

rede.adicionar_tubulacao("E-101", "C-101", 5.0, "XV-101", 2.0)
rede.adicionar_tubulacao("C-101", "TK-101", 10.0, "XV-102", 1.0)
rede.adicionar_tubulacao("C-101", "C-102", 15.0, "XV-103", 1.0)
rede.adicionar_tubulacao("C-102", "TK-102", 8.0, "XV-104", 0.5)
rede.adicionar_tubulacao("TK-101", "MAN-101", 12.0, "XV-105", 1.0)
rede.adicionar_tubulacao("TK-102", "MAN-101", 15.0, "XV-106", 0.5)
rede.adicionar_tubulacao("CH-101", "MAN-101", 6.0, "XV-107", 1.0)
rede.adicionar_tubulacao("MAN-101", "D-101", 4.0, "XV-108", 0.5)

print("Tabela de Dutos de Processo:")
print(formatar_tabela(rede.arestas_detalhes))
print("\n--- Graus Topológicos ---")
print(formatar_tabela(rede.obter_graus()))
print("\n--- Matriz de Adjacência Ponderada (Metros) ---")
print(formatar_matriz(rede.adj_pesos, rede.vertices, rede.vertices))
